<a href="https://colab.research.google.com/github/FabrizzioBurgosUni/IA/blob/main/Copia_de_Ventas_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================
# CELDA 1 - INSTALAR LIBRERÍAS
# =========================================
!pip install -q transformers accelerate bitsandbytes sentence-transformers faiss-cpu pandas torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 76.1 MB/s eta 0:00:00


# Nueva sección

In [ ]:
# =========================================
# CELDA 2 - VERIFICAR GPU (MUY IMPORTANTE)
# =========================================
import torch

if torch.cuda.is_available():
    print(f"✅ GPU disponible: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("❌ No hay GPU. Ve a: Runtime > Change runtime type > GPU")
    print("   Sin GPU este código tardará horas.")

❌ No hay GPU. Ve a: Runtime > Change runtime type > GPU
   Sin GPU este código tardará horas.


In [ ]:
# =========================================
# CELDA 3 - SUBIR Y LEER DATASET
# =========================================
from google.colab import files
import pandas as pd

uploaded = files.upload()

df = pd.read_csv('dataset_normalizado.csv')
print(f"Dataset cargado: {len(df)} filas")
print(df.head())

Saving dataset_normalizado.csv to dataset_normalizado.csv
Dataset cargado: 2823 filas
   ORDERNUMBER  QUANTITYORDERED  PRICEEACH  ORDERLINENUMBER     SALES  \
0     0.021538         0.263736   0.941193         0.058824  0.175644   
1     0.064615         0.307692   0.744940         0.235294  0.167916   
2     0.104615         0.384615   0.928063         0.058824  0.250150   
3     0.138462         0.428571   0.771061         0.294118  0.240030   
4     0.181538         0.472527   1.000000         0.764706  0.347273   

         ORDERDATE   STATUS    QTR_ID  MONTH_ID  YEAR_ID  ...  \
0   2/24/2003 0:00  Shipped  0.000000  0.090909      0.0  ...   
1    5/7/2003 0:00  Shipped  0.333333  0.363636      0.0  ...   
2    7/1/2003 0:00  Shipped  0.666667  0.545455      0.0  ...   
3   8/25/2003 0:00  Shipped  0.666667  0.636364      0.0  ...   
4  10/10/2003 0:00  Shipped  1.000000  0.818182      0.0  ...   

                    ADDRESSLINE1  ADDRESSLINE2           CITY STATE  \
0        897 

In [ ]:
# =========================================
# CELDA 4 ALTERNATIVA - DOCUMENTOS AGRUPADOS
# (más útil para preguntas analíticas)
# =========================================
import pandas as pd

documents = []

# Agrupar por PRODUCTLINE + COUNTRY + YEAR
grouped = df.groupby(['PRODUCTLINE', 'COUNTRY', 'YEAR_ID']).agg(
    SALES_TOTAL=('SALES', 'sum'),
    SALES_PROMEDIO=('SALES', 'mean'),
    CANTIDAD=('SALES', 'count'),
    DEALSIZE=('DEALSIZE', lambda x: x.mode()[0])  # moda del dealsize
).reset_index()

for _, row in grouped.iterrows():
    text = (
        f"PRODUCTLINE: {row['PRODUCTLINE']} | "
        f"COUNTRY: {row['COUNTRY']} | "
        f"YEAR_ID: {row['YEAR_ID']} | "
        f"VENTAS_TOTAL: {row['SALES_TOTAL']:.2f} | "
        f"VENTAS_PROMEDIO: {row['SALES_PROMEDIO']:.2f} | "
        f"CANTIDAD_ORDENES: {row['CANTIDAD']} | "
        f"DEALSIZE: {row['DEALSIZE']}"
    )
    documents.append(text)

print(f"✅ Dataset reducido de 28,000 → {len(documents)} documentos agrupados")

✅ Dataset reducido de 28,000 → 227 documentos agrupados


In [ ]:
# =========================================
# CELDA 5 - EMBEDDINGS + GUARDADO EN DISCO
# =========================================
import numpy as np
import os
from sentence_transformers import SentenceTransformer

EMBEDDINGS_PATH = "embeddings_28k.npy"
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

if os.path.exists(EMBEDDINGS_PATH):
    # Si ya existen, cargarlos (segundos en lugar de minutos)
    embeddings = np.load(EMBEDDINGS_PATH)
    print(f"✅ Embeddings cargados desde disco: {embeddings.shape}")
else:
    # Primera vez: generarlos y guardarlos
    print("Generando embeddings por primera vez...")
    embeddings = embedding_model.encode(
        documents,
        batch_size=128,          # 128 es óptimo para 28k
        show_progress_bar=True
    )
    np.save(EMBEDDINGS_PATH, embeddings)
    print(f"✅ Embeddings guardados: {embeddings.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generando embeddings por primera vez...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Embeddings guardados: (227, 384)


In [ ]:
# =========================================
# CELDA 6 - FAISS OPTIMIZADO PARA 28K DATOS
# =========================================
import faiss

dimension = embeddings.shape[1]  # 384

# IVFFlat: divide los vectores en clusters (mucho más rápido)
nlist = 100  # número de clusters (recomendado: sqrt(n) ≈ 167 para 28k)

quantizer = faiss.IndexFlatL2(dimension)
index = faiss.IndexIVFFlat(quantizer, dimension, nlist)

# OBLIGATORIO: entrenar el índice antes de agregar vectores
index.train(np.array(embeddings, dtype=np.float32))
index.add(np.array(embeddings, dtype=np.float32))

# nprobe: cuántos clusters revisar al buscar (balance velocidad/calidad)
index.nprobe = 10

print(f"✅ FAISS IVFFlat listo con {index.ntotal} vectores")

✅ FAISS IVFFlat listo con 227 vectores


In [ ]:
# =========================================
# CELDA 7 - MISTRAL VÍA API (sin GPU)
# =========================================
!pip install -q mistralai

from mistralai.client import Mistral

MISTRAL_API_KEY = "HF_TOKEN"  # ← Pega tu key
client = Mistral(api_key=MISTRAL_API_KEY)

print("✅ Cliente Mistral API listo")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-sdk 1.38.0 requires opentelemetry-api==1.38.0, but you have opentelemetry-api 1.39.1 which is incompatible.
opentelemetry-sdk 1.38.0 requires opentelemetry-semantic-conventions==0.59b0, but you have opentelemetry-semantic-conventions 0.60b1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.39.1 which is incompatible.
✅ Cliente Mistral API listo


In [ ]:
# =========================================
# CELDA 8 - RAG CON PROMPT MEJORADO
# =========================================
def ask_rag(question, k=5):

    # 1. Embedding
    question_embedding = embedding_model.encode([question])

    # 2. Buscar
    D, I = index.search(np.array(question_embedding, dtype=np.float32), k=k)

    # 3. Contexto
    context = "\n".join([documents[i] for i in I[0]])

    # 4. Prompt más directo y permisivo
    response = client.chat.complete(
        model="mistral-small-latest",
        messages=[
            {
                "role": "system",
                "content": """Eres un asistente de análisis de datos de ventas.
Tienes acceso a registros de ventas con campos: PRODUCTLINE, COUNTRY, YEAR_ID, SALES, DEALSIZE.
Analiza el contexto y responde de forma útil.
Si el contexto tiene información relevante, úsala aunque sea parcial."""
            },
            {
                "role": "user",
                "content": f"""Aquí están algunos registros de ventas relevantes:

{context}

Con base en estos registros, responde: {question}"""
            }
        ],
        temperature=0.3
    )

    return response.choices[0].message.content

# Test
print(ask_rag("¿Qué productos aparecen en los datos?"))

Los productos que aparecen en los datos son:

1. **Trucks and Buses**
2. **Classic Cars**


In [ ]:
# =========================================
# CELDA 9 - PROBAR EL RAG
# =========================================
respuesta = ask_rag("Qué productos tuvo mas ventas en el 2000")
print("Respuesta:", respuesta)

Respuesta: Basándome en los registros proporcionados, los productos con ventas en el año 2000 (YEAR_ID: 0.0) son:

1. **Trucks and Buses**:
   - Ventas totales: 4.32
   - Ventas promedio: 0.24
   - Cantidad de órdenes: 18
   - Tamaño de trato: Medium

2. **Classic Cars**:
   - Ventas totales: 8.48
   - Ventas promedio: 0.28
   - Cantidad de órdenes: 30
   - Tamaño de trato: Medium

**Conclusión**:
El producto con **más ventas en el año 2000** es **Classic Cars**, con un total de **8.48** en ventas, superando a **Trucks and Buses** que registró **4.32**.
